# GALVAT Trial 4 Thin Colab Workflow

This notebook matches the setup where:
- the source repo lives on your local Mac
- the dataset and Colab outputs live in Google Drive
- Colab gets the repo as a small runtime bundle zip

Order:
1. Build the Colab bundle from your Mac and make it visible in Drive.
2. Mount Drive in Colab and unpack the bundle into `/content`.
3. Generate Trial-4 prep artifacts on Drive.
4. Review the prep artifacts and create manual relabel overrides if needed.
5. Merge relabel overrides into a working labels tree on Drive.
6. Train or fine-tune on the L4 GPU.
7. Run post-train validation and save outputs back to Drive.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.0', 'pyyaml==6.0.2', 'pandas==2.2.2', 'pillow'])


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

import yaml

drive.mount('/content/drive')

# Put the Colab bundle zip from your Mac into this Drive location.
DRIVE_BUNDLE_PATH = Path('/content/drive/MyDrive/galvat_colab_bundles/ml-agent-workbench_galvat_runtime_bundle.zip')

# Dataset and all heavy outputs remain on Drive.
DATASET_ROOT = Path('/content/drive/MyDrive/galvat_openimages_v7')

# Repo bundle is unpacked into ephemeral Colab storage each session.
REPO_ROOT = Path('/content/ml-agent-workbench')
LOCAL_PREP_CACHE_ROOT = Path('/content/galvat_trial4_cache')

# Change this to a previous checkpoint if you want to continue from Trial 3.
TRAINING_MODEL = 'yolo26s.pt'
# Example:
# TRAINING_MODEL = str(DATASET_ROOT / 'experiments/colab_runs/<trial3_run_id>/models/train/weights/best.pt')

BASELINE_RUN_ID = 'galvat_run_v2_baseline_yolo26s_transfer_roi_assistive29'
PLAN_ID = 'galvat_exp_plan_v3_trial4_data_supervision_fp_control'
RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
TRIAL4_RUN_ID = f'galvat_run_v3_trial4_data_supervision_fp_control_colab_{RUN_TS}'

PREP_OUTPUT_ROOT = DATASET_ROOT / 'trial4_prep' / TRIAL4_RUN_ID
DRIVE_RUN_ROOT = DATASET_ROOT / 'experiments' / 'colab_runs' / TRIAL4_RUN_ID
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
PREP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_LABELS_ROOT = DATASET_ROOT / 'labels'
IMAGES_TRAIN_DIR = DATASET_ROOT / 'images' / 'train'
IMAGES_VAL_DIR = DATASET_ROOT / 'images' / 'validation'
IMAGES_TEST_DIR = DATASET_ROOT / 'images' / 'test'

MANUAL_RELABELS_ROOT = DATASET_ROOT / 'trial4_manual_relabels_v1'
WORKING_LABELS_ROOT = DATASET_ROOT / f'labels_trial4_working_{RUN_TS}'
VERIFIED_BACKGROUND_SLICE_PATH = DATASET_ROOT / 'trial4_verified_background_slice_v1.json'

PREP_CONFIG_PATH = Path('/content/galvat_trial4_data_prep_colab.yaml')
TRAIN_CONFIG_PATH = Path('/content/galvat_trial4_train_colab.yaml')

assert DRIVE_BUNDLE_PATH.exists(), f'Bundle zip not found: {DRIVE_BUNDLE_PATH}'
assert DATASET_ROOT.exists(), f'Dataset root not found: {DATASET_ROOT}'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
REPO_ROOT.mkdir(parents=True, exist_ok=True)
if LOCAL_PREP_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_PREP_CACHE_ROOT)
LOCAL_PREP_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE_BUNDLE_PATH, 'r') as archive:
    archive.extractall(REPO_ROOT)

assert (REPO_ROOT / 'pyproject.toml').exists(), f'pyproject.toml missing after unzip: {REPO_ROOT}'

CANONICAL_BACKGROUND_SLICE = REPO_ROOT / 'domains/galvat/datasets/galvat_background_only_roi_slice_v1.json'
EPHEMERAL_RUN_RECORD_PATH = REPO_ROOT / 'domains/galvat/experiments/runs' / f'{TRIAL4_RUN_ID}.json'
EPHEMERAL_ARTIFACT_ROOT = REPO_ROOT / 'domains/galvat/artifacts/files' / TRIAL4_RUN_ID

print('BUNDLE =', DRIVE_BUNDLE_PATH)
print('REPO_ROOT =', REPO_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('PREP_OUTPUT_ROOT =', PREP_OUTPUT_ROOT)
print('DRIVE_RUN_ROOT =', DRIVE_RUN_ROOT)
print('TRIAL4_RUN_ID =', TRIAL4_RUN_ID)
print('TRAINING_MODEL =', TRAINING_MODEL)
print('LOCAL_PREP_CACHE_ROOT =', LOCAL_PREP_CACHE_ROOT)


In [ ]:
def run_cmd(args, cwd=None, env=None):
    print('RUN', ' '.join(str(x) for x in args))
    completed = subprocess.run(args, cwd=cwd, env=env, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {args}')
    return completed.stdout

run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])


In [ ]:
prep_cfg = {
    'domain': 'galvat',
    'plan_id': PLAN_ID,
    'baseline_run_id': BASELINE_RUN_ID,
    'dataset': {
        'manifest_path': str(REPO_ROOT / 'domains/galvat/datasets/manifests/galvat_dataset_manifest_v1.json'),
        'background_slice_path': str(CANONICAL_BACKGROUND_SLICE),
        'class_support_report_path': str(REPO_ROOT / 'domains/galvat/datasets/galvat_class_support_report_v1.json'),
        'images': {
            'train': str(IMAGES_TRAIN_DIR),
            'val': str(IMAGES_VAL_DIR),
            'test': str(IMAGES_TEST_DIR),
        },
        'labels': {
            'train': str(BASE_LABELS_ROOT / 'train'),
            'val': str(BASE_LABELS_ROOT / 'validation'),
            'test': str(BASE_LABELS_ROOT / 'test'),
        },
    },
    'model_inference': {
        'weights_path': str(TRAINING_MODEL) if str(TRAINING_MODEL).endswith('.pt') else '',
    },
    'background_review': {
        'derive_from_empty_labels_if_missing': True,
        'mine_false_positives': True,
        'target_splits': ['train', 'validation', 'test'],
        'max_per_split': {'train': 1200, 'val': 250, 'test': 250},
        'conf': 0.25,
        'iou': 0.45,
        'imgsz': 640,
        'max_det': 50,
        'high_confidence_threshold': 0.50,
        'top_k': 200,
    },
    'grouped_instance': {
        'enabled': True,
        'focus_classes': ['pen', 'fork', 'spoon', 'wine_glass', 'coffee_mug', 'glasses'],
        'conf': 0.25,
        'iou': 0.45,
        'imgsz': 640,
        'max_det': 100,
        'max_images_per_class': 120,
        'min_predictions_inside_gt': 2,
        'duplicate_iou_threshold': 0.70,
        'max_prediction_area_ratio': 0.75,
        'min_coverage_ratio': 0.30,
    },
    'appearance_audit': {
        'focus_classes': ['pen', 'fork', 'spoon', 'kitchen_knife', 'knife', 'coffee_mug', 'wine_glass', 'glasses', 'remote_control'],
        'target_splits': ['train', 'validation', 'test'],
        'max_samples_per_class': 40,
    },
    'local_cache': {
        'enabled': True,
        'root_dir': str(LOCAL_PREP_CACHE_ROOT),
        'stage_labels': True,
        'stage_weights': True,
        'stage_images_for_mining': True,
    },
    'outputs': {
        'root_dir': str(PREP_OUTPUT_ROOT),
        'summary_path': str(PREP_OUTPUT_ROOT / 'summary.json'),
        'background_review_queue_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_background_review_queue_v1.json'),
        'background_fp_candidates_csv_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_background_fp_candidates_v1.csv'),
        'background_fp_candidates_json_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_background_fp_candidates_v1.json'),
        'grouped_instance_candidates_csv_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_grouped_instance_relabel_candidates_v1.csv'),
        'grouped_instance_candidates_json_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_grouped_instance_relabel_candidates_v1.json'),
        'appearance_audit_csv_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_appearance_audit_v1.csv'),
        'appearance_audit_json_path': str(PREP_OUTPUT_ROOT / 'galvat_trial4_appearance_audit_v1.json'),
    },
}

PREP_CONFIG_PATH.write_text(yaml.safe_dump(prep_cfg, sort_keys=False), encoding='utf-8')
print(PREP_CONFIG_PATH.read_text(encoding='utf-8'))

run_cmd([sys.executable, str(REPO_ROOT / 'domains/galvat/runtime/scripts/prepare_trial4_data.py'), '--config', str(PREP_CONFIG_PATH)])
print('Prep outputs written to:', PREP_OUTPUT_ROOT)


## Manual Review Gate

Stop here after the prep script finishes.

Review these Drive artifacts:
- `PREP_OUTPUT_ROOT / galvat_trial4_background_review_queue_v1.json`
- `PREP_OUTPUT_ROOT / galvat_trial4_background_fp_candidates_v1.csv`
- `PREP_OUTPUT_ROOT / galvat_trial4_grouped_instance_relabel_candidates_v1.csv`
- `PREP_OUTPUT_ROOT / galvat_trial4_appearance_audit_v1.csv`

Then do the following before training:
- Create manual relabel overrides in `galvat_openimages_v7/trial4_manual_relabels_v1/<split>/<image_id>.txt`
- If you curate a verified background slice, save it to `galvat_openimages_v7/trial4_verified_background_slice_v1.json`
- Re-run the remaining cells after those edits are ready


In [ ]:
override_txt_files = []
if MANUAL_RELABELS_ROOT.exists():
    override_txt_files = list(MANUAL_RELABELS_ROOT.rglob('*.txt'))

if override_txt_files:
    run_cmd([
        sys.executable,
        str(REPO_ROOT / 'domains/galvat/runtime/scripts/merge_label_overrides.py'),
        '--base-labels-root', str(BASE_LABELS_ROOT),
        '--override-labels-root', str(MANUAL_RELABELS_ROOT),
        '--output-labels-root', str(WORKING_LABELS_ROOT),
        '--clean-output',
    ])
    ACTIVE_LABELS_ROOT = WORKING_LABELS_ROOT
else:
    ACTIVE_LABELS_ROOT = BASE_LABELS_ROOT
    print('No manual relabel overrides found. Using base labels root.')

ACTIVE_BACKGROUND_SLICE = VERIFIED_BACKGROUND_SLICE_PATH if VERIFIED_BACKGROUND_SLICE_PATH.exists() else CANONICAL_BACKGROUND_SLICE

print('ACTIVE_LABELS_ROOT =', ACTIVE_LABELS_ROOT)
print('ACTIVE_BACKGROUND_SLICE =', ACTIVE_BACKGROUND_SLICE)


In [ ]:
run_record = {
    'run_id': TRIAL4_RUN_ID,
    'experiment_name': 'trial4_data_supervision_fp_control_colab',
    'domain': 'galvat',
    'status': 'planned',
    'objective': 'Reduce background false positives and label-noise-driven over-detection using a data supervision package while keeping the YOLO26 training recipe stable.',
    'hypothesis': 'Curated background negatives, grouped-instance relabels, and improved appearance coverage will reduce background confusion and repeated-object annotation mismatch without a model-family change.',
    'baseline_run_id': BASELINE_RUN_ID,
    'tags': [
        f'plan_id:{PLAN_ID}',
        'one_major_variable:data_supervision_only',
        'trial4',
        'background_fp_control',
        'grouped_instance_relabel',
        'appearance_robustness',
    ],
    'metrics': {},
    'artifacts': [],
    'notes': '',
    'created_at': datetime.now(timezone.utc).date().isoformat(),
}
EPHEMERAL_RUN_RECORD_PATH.write_text(json.dumps(run_record, indent=2), encoding='utf-8')

train_cfg = {
    'domain': 'galvat',
    'run_id': TRIAL4_RUN_ID,
    'plan_id': PLAN_ID,
    'paths': {
        'dataset_manifest_path': str(REPO_ROOT / 'domains/galvat/datasets/manifests/galvat_dataset_manifest_v1.json'),
        'ontology_mapping_table_path': str(REPO_ROOT / 'domains/galvat/datasets/galvat_ontology_mapping_table_v1.json'),
        'scene_split_registry_path': str(REPO_ROOT / 'domains/galvat/datasets/galvat_scene_split_registry_v1.json'),
        'background_slice_path': str(ACTIVE_BACKGROUND_SLICE),
        'hardware_harness_path': str(REPO_ROOT / 'domains/galvat/runtime/configs/galvat_fixed_hardware_eval_harness_v1.yaml'),
        'labels_root': str(ACTIVE_LABELS_ROOT),
        'output_root': str(DRIVE_RUN_ROOT),
    },
    'images': {
        'train_dir': str(IMAGES_TRAIN_DIR),
        'val_dir': str(IMAGES_VAL_DIR),
        'test_dir': str(IMAGES_TEST_DIR),
    },
    'subset': {
        'mode': 'schema_order',
        'expected_active_class_count': 29,
    },
    'training': {
        'model': str(TRAINING_MODEL),
        'epochs': 100,
        'imgsz': 640,
        'batch': 16,
        'device': '0',
        'workers': 8,
    },
    'onnx_export': {
        'opset': 13,
        'simplify': True,
    },
}

TRAIN_CONFIG_PATH.write_text(yaml.safe_dump(train_cfg, sort_keys=False), encoding='utf-8')
print(TRAIN_CONFIG_PATH.read_text(encoding='utf-8'))


In [ ]:
run_cmd([sys.executable, str(REPO_ROOT / 'domains/galvat/runtime/scripts/train.py'), '--config', str(TRAIN_CONFIG_PATH)])

DRIVE_REPO_STATE_ROOT = DRIVE_RUN_ROOT / 'repo_state'
DRIVE_REPO_STATE_ROOT.mkdir(parents=True, exist_ok=True)

if EPHEMERAL_RUN_RECORD_PATH.exists():
    shutil.copy2(EPHEMERAL_RUN_RECORD_PATH, DRIVE_REPO_STATE_ROOT / 'run_record.json')
if EPHEMERAL_ARTIFACT_ROOT.exists():
    target = DRIVE_REPO_STATE_ROOT / 'artifacts_files'
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(EPHEMERAL_ARTIFACT_ROOT, target)
shutil.copy2(TRAIN_CONFIG_PATH, DRIVE_REPO_STATE_ROOT / 'train_config.yaml')
shutil.copy2(PREP_CONFIG_PATH, DRIVE_REPO_STATE_ROOT / 'prep_config.yaml')

summary_candidates = [
    EPHEMERAL_ARTIFACT_ROOT / 'train_execution_summary.json',
    DRIVE_RUN_ROOT / 'train_execution_summary.json',
]
for candidate in summary_candidates:
    if candidate.exists():
        print(candidate.read_text(encoding='utf-8'))
        break


In [ ]:
from ultralytics import YOLO

dataset_yaml_path = DRIVE_RUN_ROOT / 'dataset_yolo26_curated29.yaml'
best_weights_path = DRIVE_RUN_ROOT / 'yolo_runs' / TRIAL4_RUN_ID / 'weights' / 'best.pt'
post_eval_root = DRIVE_RUN_ROOT / 'posttrain_eval'
post_eval_root.mkdir(parents=True, exist_ok=True)

best_model = YOLO(str(best_weights_path))

val_result = best_model.val(
    data=str(dataset_yaml_path),
    split='val',
    imgsz=640,
    project=str(post_eval_root),
    name='val',
    exist_ok=True,
    plots=True,
)
test_result = best_model.val(
    data=str(dataset_yaml_path),
    split='test',
    imgsz=640,
    project=str(post_eval_root),
    name='test',
    exist_ok=True,
    plots=True,
)

def clean_metrics(result):
    metrics = {}
    result_dict = getattr(result, 'results_dict', None)
    if isinstance(result_dict, dict):
        for key, value in result_dict.items():
            if isinstance(value, (int, float, bool, str)) or value is None:
                metrics[str(key)] = value
    return metrics

val_metrics = clean_metrics(val_result)
test_metrics = clean_metrics(test_result)

(DRIVE_RUN_ROOT / 'val_metrics_summary.json').write_text(json.dumps(val_metrics, indent=2), encoding='utf-8')
(DRIVE_RUN_ROOT / 'test_metrics_summary.json').write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')

print('VAL METRICS')
print(json.dumps(val_metrics, indent=2))
print('TEST METRICS')
print(json.dumps(test_metrics, indent=2))
print('Confusion matrix folder:', post_eval_root / 'val')


## Correct Order Every Time

Local Mac:
1. Update the repo.
2. Build a fresh Colab bundle zip from the repo.
3. Put that zip in your Drive bundle folder.

Colab:
1. Run the bundle-unpack cell.
2. Run the prep cell.
3. Review prep artifacts in Drive.
4. Add manual relabel overrides and, if needed, a verified background slice in Drive.
5. Re-run the merge/train config cells.
6. Train.
7. Run post-train validation.

If you still want the older ROI-boundary, spoken-burden, or split-integrity diagnostics, run sections 13-16 from `galvat_v7_assistive29_colab_train.ipynb` against the new `best.pt` and save those outputs into `DRIVE_RUN_ROOT`.
